# Import the libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import yfinance as yf
import plotly.express as px


from pandas.plotting import register_matplotlib_converters
register_matplotlib_converters()

# Efficient Frontier

Investors have different appetites for risk

Some favour safer returns at the expense of less change of exceptional returns.

Portfolio theory is that some risk can be mitigated by mixing ecurities together

Can allow investors to earn higher rate of return whilst reducing risk

## Load in some Data

1 years worth

Approx 252 days

In [ ]:
stocks = 'JPM IBM'
stocks.split()

In [ ]:
stocks = 'AAPL GLD'.split()
df = yf.download(tickers=stocks, start='2022-01-01', auto_adjust=True)['Close']
df = df[-253:]

df_vol = pd.DataFrame()

for stock in stocks:
    if stock not in df_vol:
        df_vol[stock] = np.log(df[stock]).diff() 

In [ ]:
df_vol


## Annualize variances

Multiply daily variance by 252

In [ ]:
var_aapl = df_vol['AAPL'].var() * 252
var_gld = df_vol['GLD'].var() * 252
print(var_aapl)
print(var_gld)

## Construct a Portfolio

Weights - 90% apple, 10% gold

Expected Returns -  (Apple 14%), Gold(7%)

In [ ]:
w_aapl =  .9
w_gld = 1 - w_aapl
exp_aapl = .14
exp_gld = .07

In [ ]:
exp = w_aapl * exp_aapl + w_gld * exp_gld
exp

### Anualize the covariance

In [ ]:
np.cov(df_vol['AAPL'][1:], df_vol['GLD'][1:])[1,0]

In [ ]:
cov = np.cov(df_vol['AAPL'][1:], df_vol['GLD'][1:])[0,1] * 252
cov

### Calculate std of Portfolio

Based on weights (90% & 10%)

$ \large std_{port} = \sqrt{var_{aapl}.(weight_{aapl})^2 + var_{gld}.(weight_{gld})^2 + 2. cov.weight_{aapl}.weight_{gld} }$

In [ ]:
port_std = np.sqrt(var_aapl * w_aapl **2 + var_gld * w_gld ** 2 + 2 * cov * w_aapl * w_gld)
port_std

## Analyse Results

With 90% Apple & 10% gold

- Expect return is 13.3%
- Expected volatility is 36.7%

Might consider this too high a risk

## Repeat

But with difference weights

e.g. weights of 0%, 5%, 10%, 155, ... 100%

In [ ]:
df_effic = pd.DataFrame({'weight_aapl':np.zeros(21), 'exp_ret': np.zeros(21), 'std': np.zeros(21)})

df_effic

In [ ]:
df_effic = pd.DataFrame({'weight_aapl': np.zeros(21),
                         'exp_ret': np.zeros(21),
                         'std': np.zeros(21)})

w_aapl = 0.0
for weight in range(21):
    df_effic.loc[weight, 'weight_aapl'] = w_aapl
    df_effic.loc[weight, 'exp_ret'] = w_aapl * exp_aapl + (1 - w_aapl) * exp_gld
    df_effic.loc[weight, 'std'] = np.sqrt(
        var_aapl * w_aapl**2
        + var_gld * (1 - w_aapl)**2
        + cov * w_aapl * (1 - w_aapl)
    )
    w_aapl += 0.05

df_effic

In [ ]:
# Scatter plot of risk (std) vs return
fig = px.scatter(
    df_effic,
    x="std",
    y="exp_ret",
    title="Efficient Frontier (AAPL vs GLD)",
    labels={"std": "Portfolio Risk (Std Dev)", "exp_ret": "Expected Return"},
)

# Optional: make it look more like seaborn style
fig.update_layout(
    template="seaborn",  # Plotly has a seaborn-style template
    width=800,
    height=500
)

fig.show()